In [1]:
# ===== SETUP: Run this cell first! =====
# All imports and constants needed throughout the notebook

import numpy as np
import tiktoken
import time

# Hyperparameters / Constants
D_MODEL = 128          # Embedding dimension (common: 128, 256, 512, 768)
DISPLAY_DP = 2         # Decimal places for display

# Seed for reproducibility
np.random.seed(42)

print("✓ Imports and constants loaded")
print(f"  - D_MODEL: {D_MODEL}")
print(f"  - DISPLAY_DP: {DISPLAY_DP}")
print(f"  - NumPy random seed: 42")

✓ Imports and constants loaded
  - D_MODEL: 128
  - DISPLAY_DP: 2
  - NumPy random seed: 42


# ZachGPT - Learning LLMs from Scratch

## Tokenization

What is a token?

Simply a token is an index that we use to represent a word, subword or character.

eg. My House -> 4, 7

Where there is a Map from My to 4 and house to 7

Early Models look at just the 26 characters, some try lots of words and more recently a list of subword's, words and characters.
A subword is a common set of characters eg. Unknown may be two subwords un and known.

There are a few different algorithms for tokenising and deciding what subwords to use so you have a common set of words but you dont miss anything and you dont have to many words whereby the vocab size actually starts giving worse results.

An example algorithm is Byte Pair Encoding
You start with characters in your list of tokens and a bunch of text to run the algo on.

You find the most common pair of characters in your text and add this to your list of tokens. Then you do this until you have the desired number of tokens common amounts are around 32'000 -> 50'000.
When doing this and When tokenising it is common employ a rule of longest match first. So if I have tokens u=1 , n=2 and un =3 the start of unknown becomes 3.


"From Scratch caveat"
Im not writing the tokeniser I will be using the GPT-2 Tokeniser that has decided the tokens for me. This is because tokenization is not all that interesting.

In [2]:
import tiktoken

class Tokenizer:
    """Wrapper around tiktoken's GPT-2 BPE encoding"""

    def __init__(self, encoding_name: str = "gpt2"):
        self.encoding = tiktoken.get_encoding(encoding_name)

    def encode(self, text: str) -> list[int]:
        """Convert text to token IDs"""
        return self.encoding.encode(text)

    def decode(self, tokens: list[int] | int) -> str:
        """Convert token IDs back to text"""
        if isinstance(tokens, int):
            tokens = [tokens]
        return self.encoding.decode(tokens)

    def encode_batch(self, texts: list[str]) -> list[list[int]]:
        """Encode multiple texts"""
        return [self.encode(text) for text in texts]

    @property
    def vocab_size(self) -> int:
        """Return vocabulary size"""
        return self.encoding.n_vocab

    @property
    def eot_token(self) -> int:
        """End of text token ID"""
        return self.encoding.eot_token

tokenizer = Tokenizer()
print(f"Vocabulary size: {tokenizer.vocab_size}")

text = "Hello, world! This is ZachGPT learning to tokenize."
tokens = tokenizer.encode(text)

print(f"Original text: {text}")
print(f"\nToken IDs: {tokens}")
print(f"Number of tokens: {len(tokens)}")
print(f"\nDecoded back: {tokenizer.decode(tokens)}")

print("Token breakdown:")
for i, token_id in enumerate(tokens):
    token_text = tokenizer.decode(token_id)
    print(f"  {i}: ID={token_id:5d} → '{token_text}'")

print(f"Special End of Text (EOT) marks boundary's as training is done in sentences token ID: {tokenizer.eot_token}")

Vocabulary size: 50257
Original text: Hello, world! This is ZachGPT learning to tokenize.

Token IDs: [15496, 11, 995, 0, 770, 318, 18825, 38, 11571, 4673, 284, 11241, 1096, 13]
Number of tokens: 14

Decoded back: Hello, world! This is ZachGPT learning to tokenize.
Token breakdown:
  0: ID=15496 → 'Hello'
  1: ID=   11 → ','
  2: ID=  995 → ' world'
  3: ID=    0 → '!'
  4: ID=  770 → ' This'
  5: ID=  318 → ' is'
  6: ID=18825 → ' Zach'
  7: ID=   38 → 'G'
  8: ID=11571 → 'PT'
  9: ID= 4673 → ' learning'
  10: ID=  284 → ' to'
  11: ID=11241 → ' token'
  12: ID= 1096 → 'ize'
  13: ID=   13 → '.'
Special End of Text (EOT) marks boundary's as training is done in sentences token ID: 50256


## Embedding
Converting the tokens to an n dimensional array in this case 128 dimensions.

You initialize these with random values before training. We are using a normal distribution which means most of our values are between -0.6 - 0.6. This helps with training by not having as much sway in backpropagation

In [3]:
VOCAB_SIZE = tokenizer.vocab_size  # 50257 from GPT-2 tokenizer

# Initialize embedding matrix with random values
# Shape: (vocab_size, d_model) - one d_model-dimensional vector per token
# Using normal distribution: mean=0, std=0.02 (same as GPT-2)
embedding_matrix = np.random.normal(loc=0.0, scale=0.02, size=(VOCAB_SIZE, D_MODEL))

print(f"Embedding Matrix Shape: {embedding_matrix.shape}")
print(f"  - {VOCAB_SIZE:,} tokens (vocabulary size)")
print(f"  - {D_MODEL} dimensions per token")
print(f"\nMemory usage: {embedding_matrix.nbytes / 1024 / 1024:.2f} MB")
print(f"\nInitialization stats:")
print(f"  Mean: {embedding_matrix.mean():.6f}")
print(f"  Std:  {embedding_matrix.std():.6f}")
print(f"  Min:  {embedding_matrix.min():.6f}")
print(f"  Max:  {embedding_matrix.max():.6f}")

# What token ID does "Hello" map to?
hello_token_id = tokenizer.encode("Hello")[0]          # 15496
hello_embedding = embedding_matrix[hello_token_id]

print(f"\nExample - Embedding for 'Hello' (token ID {hello_token_id}):")
print(f" Full Vector: {np.round(hello_embedding[:10], DISPLAY_DP)}")

Embedding Matrix Shape: (50257, 128)
  - 50,257 tokens (vocabulary size)
  - 128 dimensions per token

Memory usage: 49.08 MB

Initialization stats:
  Mean: -0.000004
  Std:  0.020001
  Min:  -0.099603
  Max:  0.104401

Example - Embedding for 'Hello' (token ID 15496):
 Full Vector: [ 0.01  0.04 -0.    0.03  0.   -0.02  0.02  0.02  0.01  0.02]




## Positional Embedding

Positional embeddings are vectors added to the token embeddings to inject order. Each position (0, 1, 2, ...) gets its own vector:

```
final_embedding = token_embedding + positional_embedding
```

Understanding position in a sentence is crucial as the meaning of a sentence can completely change despite the words being the same. Take for example "the cat sat upon the hat" and "the hat sat upon the cat" the position of the words changes the whole meaning of the sentence.
We use embeddings for this as there is no mechanism to learn position in the algorithm itself, see attention for more info on this.

We use the sinusoidal pattern from the original "Attention Is All You Need" paper. Each dimension alternates between sin and cos at different frequencies, giving every position a unique vector where nearby positions have similar vectors. Crucially these are fixed (not learned) so the model can generalise to sequences longer than it was trained on.

In [4]:
MAX_SEQ_LEN = 512  # maximum number of tokens we'll ever feed in at once

def positional_encoding(max_seq_len: int, d_model: int) -> np.ndarray:
    """
    Build a (max_seq_len, d_model) matrix where each row is the
    positional encoding for that position.

    For each position pos and each dimension i:
      PE[pos, 2i]   = sin(pos / 10000^(2i/d_model))
      PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
    """
    PE = np.zeros((max_seq_len, d_model))
    positions = np.arange(max_seq_len).reshape(-1, 1)          # (max_seq_len, 1)
    dims      = np.arange(0, d_model, 2)                       # even indices: 0, 2, 4, ...
    freqs     = 1 / (10000 ** (dims / d_model))                # one frequency per dim pair

    PE[:, 0::2] = np.sin(positions * freqs)   # even dimensions → sin
    PE[:, 1::2] = np.cos(positions * freqs)   # odd  dimensions → cos
    return PE

positional_matrix = positional_encoding(MAX_SEQ_LEN, D_MODEL)

print(f"Positional Encoding Matrix Shape: {positional_matrix.shape}")
print(f"  - {MAX_SEQ_LEN} positions (max sequence length)")
print(f"  - {D_MODEL} dimensions per position")

# --- Apply to our sentence ---
# Get token embeddings for all tokens in our sentence
token_embeddings = embedding_matrix[tokens]            # shape: (14, 128)

# Get positional encodings for positions 0..13
seq_len = len(tokens)
pos_encodings = positional_matrix[:seq_len]            # shape: (14, 128)

# Add them together — now each vector encodes both *what* and *where*
final_embeddings = token_embeddings + pos_encodings    # shape: (14, 128)

print(f"\nSentence: '{text}'")
print(f"  token_embeddings shape: {token_embeddings.shape}")
print(f"  pos_encodings shape:    {pos_encodings.shape}")
print(f"  final_embeddings shape: {final_embeddings.shape}")

# Show first 5 tokens, one per line, with 4 decimal places so positional values are visible
print(f"\n{'Pos':<4} {'Token':<12} {'Token Embedding (first 5 dims)':<40} {'Positional Encoding (first 5 dims)':<40} {'Final (sum)'}")
print("-" * 130)
for i in range(5):
    token_text = tokenizer.decode(tokens[i])
    te  = np.round(token_embeddings[i, :5], 4)
    pe  = np.round(pos_encodings[i, :5], 4)
    fe  = np.round(final_embeddings[i, :5], 4)
    print(f"{i:<4} {repr(token_text):<12} {str(te):<40} {str(pe):<40} {fe}")

Positional Encoding Matrix Shape: (512, 128)
  - 512 positions (max sequence length)
  - 128 dimensions per position

Sentence: 'Hello, world! This is ZachGPT learning to tokenize.'
  token_embeddings shape: (14, 128)
  pos_encodings shape:    (14, 128)
  final_embeddings shape: (14, 128)

Pos  Token        Token Embedding (first 5 dims)           Positional Encoding (first 5 dims)       Final (sum)
----------------------------------------------------------------------------------------------------------------------------------
0    'Hello'      [ 0.0091  0.0354 -0.0013  0.0252  0.0007] [0. 1. 0. 1. 0.]                         [ 9.1000e-03  1.0354e+00 -1.3000e-03  1.0252e+00  7.0000e-04]
1    ','          [-0.0065 -0.005   0.0266  0.0111  0.0091] [0.8415 0.5403 0.7617 0.6479 0.6816]     [0.835  0.5353 0.7883 0.659  0.6907]
2    ' world'     [ 0.0202  0.014  -0.0331  0.0215 -0.0282] [ 0.9093 -0.4161  0.987  -0.1604  0.9975] [ 0.9295 -0.4021  0.9539 -0.139   0.9693]
3    '!'          [ 0

## Attention

We will talk about Attention in 2 layers of abstraction,
Firstly as an abstract way to think, Attention is the method by which the the model is choosing what to focus on. So in the sentence "The Cat is Sat on the Hill". The words cat sat and hill are more important that the words the and is.

Secondly what really happens.
Attention is designed to deal with an issue of human language where any word in a sentence could drastically affect the meaning of that sentence or the meaning of any other word in the sentence.

For example the word mole has 2 different meanings in
"a mole in the ground"
and
"a mole in a government"

As such we need a way for the word "ground" to impact the word "mole" so we get to the real meaning of mole in this case.

Attention is the mechanism that allows the embedded value to impact the other embedded values.

Note position is also in our token embedding so attention can understand that in these two sentences
The cat is on the hat
and
The hat is on the cat
Cat changes the value of the embedding of "hat" (attends to hat) in a different way in each sentence.


## How Attention Actually Works - The Full Picture

### First: How Matrix Multiplication Works

Before anything else, matrix multiplication is the core operation everywhere in attention. The rule is simple:

```
[A, B] @ [B, C] = [A, C]
```

The inner two numbers must match and they cancel. The outer two survive as the output shape.

What is actually happening is the second matrix is applied independently to every row of the first matrix. So if you have 14 tokens and multiply by a weight matrix, each of the 14 rows gets transformed by that same weight matrix independently. The 14 just rides along.

```
[14, 128]  @  [128, 32]  =  [14, 32]
    ↑               ↑            ↑
14 tokens     inner dims    14 tokens
              must match    now 32 wide
              (both 128)
              and cancel
```

---

### Step 1 - You Already Have This: final_embeddings [14, 128]

Coming out of positional encoding we have one 128-dimensional vector per token.

```
         128 dimensions
        ←————————————————→
      ┌──────────────────────┐
  'Hello'  │  0.01  1.04  -0.00  ...  │  row 0
      ├──────────────────────┤
  ','      │  0.84   0.54   0.79  ...  │  row 1
      ├──────────────────────┤
  ' world' │  0.93  -0.40   0.95  ...  │  row 2
      ├──────────────────────┤
  ...      │         ...               │
      └──────────────────────┘
             14 rows (one per token)
```

At this point every token is its own island. No token has any awareness of any other token.

---

### Step 2 - Project to Q, K, V (still no interaction)

We create three learned weight matrices and multiply our embeddings through each one.

```
W_Q shape: [128, 32]
W_K shape: [128, 32]
W_V shape: [128, 32]
```

```
Q = final_embeddings @ W_Q       [14, 128] @ [128, 32] = [14, 32]
K = final_embeddings @ W_K       [14, 128] @ [128, 32] = [14, 32]
V = final_embeddings @ W_V       [14, 128] @ [128, 32] = [14, 32]
```

Same input, three different transformations. Every token now has three 32-dimensional vectors. Token 1 still has not seen token 2. These are just three different lenses on the same data.

Why three? Q is "what am I looking for", K is "what do I advertise myself as", V is "what I actually give out". Keeping them separate lets the model learn these three roles independently. If they were all the same matrix those roles would be tangled together and the model would be less expressive.

---

### Step 3 - Scores Matrix: This Is Where Tokens First Interact

```
scores = Q @ Kᵀ                  [14, 32] @ [32, 14] = [14, 14]
```

The Kᵀ just means K transposed - flipping it from [14, 32] to [32, 14] so the matrix multiplication works.

The output is a [14, 14] matrix. This is the most important matrix in the whole operation.

```
                    K of token:
                  0    1    2  ...  13
                ┌─────────────────────┐
Q of token 0  → │ 2.1  0.3  1.8  ...  │  how much should token 0 attend to each token?
Q of token 1  → │ 0.1  3.2  0.4  ...  │  how much should token 1 attend to each token?
Q of token 2  → │ 1.4  0.2  2.9  ...  │  how much should token 2 attend to each token?
...           → │        ...          │
Q of token 13 → │ 0.8  1.1  0.3  ...  │
                └─────────────────────┘
                        [14, 14]
```

Entry [2, 5] = token 2's Q dotted with token 5's K = "how much should token 2 pay attention to token 5"

Notice this matrix is NOT symmetric. Entry [2, 5] and entry [5, 2] are different numbers:
```
[2, 5] = Q[2] · K[5]    token 2 looking for token 5
[5, 2] = Q[5] · K[2]    token 5 looking for token 2
```
"sat" might strongly attend to "cat" without "cat" strongly attending back to "sat". That asymmetry is what lets attention model directed relationships in language.

---

### Step 4 - Scale and Softmax

```
scores = scores / sqrt(32)       still [14, 14]
weights = softmax(scores, axis=-1)   still [14, 14]
```

Dividing by sqrt(64) stops the dot products getting too large as dimensions grow, which would make softmax output near-zero gradients and slow training.

Softmax converts each row into percentages that sum to 1. Now each row is "what fraction of my attention goes to each token".

```
                  after softmax, each row sums to 1.0
                ┌──────────────────────────────────┐
token 0's row → │ 0.05  0.02  0.40  0.01 ...  0.08 │  = 1.0
token 1's row → │ 0.01  0.60  0.05  0.10 ...  0.03 │  = 1.0
...           → │              ...                  │
                └──────────────────────────────────┘
```

---

### Step 5 - Weighted Sum of Values

```
output = weights @ V             [14, 14] @ [14, 32] = [14, 32]
```

Each token's output is a blend of all V vectors weighted by the attention percentages from step 4. If token 2 gave 40% attention to token 5 and 30% to token 0, its output will be 40% of token 5's V vector plus 30% of token 0's V vector plus smaller contributions from the rest.

Every token has now absorbed information from every other token. Token 2's output vector is no longer just about token 2 - it carries context from the whole sentence.

---

### In short

3 learned weights
```
W_Q shape: [128, 32]
W_K shape: [128, 32]
W_V shape: [128, 32]
```

Matrix Multiplied with our embedded tokens (the combination of positional and standard embeddings)
```
Q = final_embeddings @ W_Q       [14, 128] @ [128, 32] = [14, 32]
K = final_embeddings @ W_K       [14, 128] @ [128, 32] = [14, 32]
V = final_embeddings @ W_V       [14, 128] @ [128, 32] = [14, 32]
```

Connected together as Q multiplied by the transform of K so they can fix together. Normalised and multiplied by V.
```
 softmax((Q @ Kᵀ) / √d_k) @ V
```

Notice that these matrix multiplication's contain the number of tokens are sentence is which creates the big O notation for inference n^2.

---

### Step 6 - Multi-Head Attention: Getting Back to [14, 128]

One head projected us down from 128 to 32 dimensions. We run multiple heads in parallel each with their own W_Q, W_K, W_V and then concatenate the outputs back together.

With 4 heads:
```
head_1 output:  [14, 32]
head_2 output:  [14, 32]
head_3 output:  [14, 32]
head_4 output:  [14, 32]

concatenated:   [14, 128]   ← back to original size
```

Then one final projection W_O of shape [128, 128] mixes the heads together:
```
output = concatenated @ W_O    [14, 128] @ [128, 128] = [14, 128]
```

We are back to exactly [14, 128] - same shape as what went in. Each token is now contextually enriched by every other token but the shape is preserved so it can feed straight into the next layer.

Multiple heads matter because each head can learn to pick up on different kinds of relationships - one head might learn grammar structure, another might learn which pronouns refer to which nouns. They run in parallel for free since they are independent.

---

### Step 7 - Predicting the Next Token: W_vocab

Attention gives us back [14, 128] - contextually enriched vectors, but still in embedding space. To actually predict what word comes next we need to project back up to vocabulary size so we get a score for every possible token.

```
W_vocab shape: [128, 50257]
```

We only care about the last token's vector (the one that has seen everything before it):

```
attention_output[-1]      [128]              ← last token's contextual vector
        ↓  @ W_vocab      [128, 50257]
logits                    [50257]            ← one score per vocab token
        ↓  softmax
probabilities             [50257]            ← pick the highest one
```

Each of the 50,257 output numbers is the model's score for that token being the next one. Softmax turns these into probabilities and we pick the highest (or sample from the distribution for more varied text).

This is another learned weight matrix. It is essentially the reverse of the embedding matrix - the embedding matrix converts a token ID into a 128-dim vector, and W_vocab converts a 128-dim vector back into a score for each token ID.

---

### The Full Shape Journey

```
final_embeddings          [14, 128]
        ↓  @ W_Q/W_K/W_V [128, 32]
Q, K, V                   [14,  32]   (×3, no interaction yet)
        ↓  Q @ Kᵀ
scores                    [14,  14]   ← first token interaction
        ↓  scale + softmax
weights                   [14,  14]   (rows sum to 1)
        ↓  weights @ V
single head output        [14,  32]
        ↓  concat 4 heads
concatenated              [14, 128]
        ↓  @ W_O [128, 128]
attention output          [14, 128]   ← same shape as input
        ↓  take last token's vector
last token vector         [128]
        ↓  @ W_vocab [128, 50257]
logits                    [50257]     ← score for every vocab token
        ↓  softmax + argmax
predicted next token      scalar      ← one token ID
```

The sentence went in as 14 independent vectors. It came out as 14 vectors that each carry the full context of the sentence baked in. Then the last vector gets projected to a prediction over the entire vocabulary.

---
### Learning

This gives us many things to learn
```
  1. Embedding matrix — [50257, 128] — what each token means
  2. W_Q — [128, 32] — what each token is looking for        (×4 heads)
  3. W_K — [128, 32] — what each token advertises itself as   (×4 heads)
  4. W_V — [128, 32] — what each token gives out              (×4 heads)
  5. W_O — [128, 128] — how to mix the heads together
  6. W_vocab — [128, 50257] — convert back to a token prediction
```
W_Q, W_K, W_V are per head so with 4 heads that is 12 weight matrices, plus W_O, W_vocab, and the embedding matrix.

## Attention in Code

Now lets implement everything described above. We initialise W_Q, W_K, W_V for each head and W_O with random weights. In a real model these would be learned during training but we can still run the full forward pass to see the mechanics working.

In [5]:
N_HEADS = 4
D_HEAD = D_MODEL // N_HEADS  # 128 // 4 = 32

def softmax(x, axis=-1):
    """Numerically stable softmax"""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

# --- Initialise weight matrices (random, would be learned in training) ---

# Per-head weights: W_Q, W_K, W_V each [128, 32] for each of 4 heads
W_Q = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_K = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_V = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]

# Output projection: W_O [128, 128]
W_O = np.random.normal(0, 0.02, (D_MODEL, D_MODEL))

print(f"Hyperparameters:")
print(f"  N_HEADS: {N_HEADS}")
print(f"  D_HEAD:  {D_HEAD}")
print(f"\nWeight shapes (per head):")
print(f"  W_Q: {W_Q[0].shape}  (×{N_HEADS} heads)")
print(f"  W_K: {W_K[0].shape}  (×{N_HEADS} heads)")
print(f"  W_V: {W_V[0].shape}  (×{N_HEADS} heads)")
print(f"  W_O: {W_O.shape}     (shared across heads)")

total_params = N_HEADS * 3 * D_MODEL * D_HEAD + D_MODEL * D_MODEL
print(f"\nTotal attention parameters: {total_params:,}")

Hyperparameters:
  N_HEADS: 4
  D_HEAD:  32

Weight shapes (per head):
  W_Q: (128, 32)  (×4 heads)
  W_K: (128, 32)  (×4 heads)
  W_V: (128, 32)  (×4 heads)
  W_O: (128, 128)     (shared across heads)

Total attention parameters: 65,536


In [6]:
# --- Run multi-head attention on our sentence ---

head_outputs = []

for h in range(N_HEADS):
    # Step 2: Project to Q, K, V
    Q = final_embeddings @ W_Q[h]   # [14, 128] @ [128, 32] = [14, 32]
    K = final_embeddings @ W_K[h]   # [14, 128] @ [128, 32] = [14, 32]
    V = final_embeddings @ W_V[h]   # [14, 128] @ [128, 32] = [14, 32]

    # Step 3: Scores - tokens interact for the first time
    scores = Q @ K.T                # [14, 32] @ [32, 14] = [14, 14]

    # Step 4: Scale and softmax
    scores = scores / np.sqrt(D_HEAD)
    weights = softmax(scores)       # each row sums to 1.0

    # Step 5: Weighted sum of values
    head_out = weights @ V          # [14, 14] @ [14, 32] = [14, 32]
    head_outputs.append(head_out)

    if h == 0:
        print(f"Head {h} walkthrough:")
        print(f"  Q shape: {Q.shape}")
        print(f"  K shape: {K.shape}")
        print(f"  V shape: {V.shape}")
        print(f"  scores shape: {scores.shape}")
        print(f"  weights shape: {weights.shape}  (row sums: {np.round(weights.sum(axis=1)[:3], 4)}...)")
        print(f"  head output shape: {head_out.shape}")
        print()

# Step 6: Concatenate heads and project with W_O
concatenated = np.concatenate(head_outputs, axis=1)  # [14, 128]
attention_output = concatenated @ W_O                 # [14, 128] @ [128, 128] = [14, 128]

print(f"Concatenated {N_HEADS} heads: {concatenated.shape}")
print(f"After W_O projection:         {attention_output.shape}")
print(f"\nInput shape:  {final_embeddings.shape}")
print(f"Output shape: {attention_output.shape}  ← same shape, now contextually enriched")

Head 0 walkthrough:
  Q shape: (14, 32)
  K shape: (14, 32)
  V shape: (14, 32)
  scores shape: (14, 14)
  weights shape: (14, 14)  (row sums: [1. 1. 1.]...)
  head output shape: (14, 32)

Concatenated 4 heads: (14, 128)
After W_O projection:         (14, 128)

Input shape:  (14, 128)
Output shape: (14, 128)  ← same shape, now contextually enriched


## The Transformer Block

At this point we could connect the attention output directly to W_vocab and get predictions — in fact that is exactly what the original attention demo above does. But going straight from attention to prediction is extremely limiting. Attention blends information between tokens but there is no step where the model actually *processes* or "thinks about" the blended result. It would be like reading a sentence and having to answer a question about it before you have had time to think.

By adding layers between attention and the prediction we give the model capacity for deeper understanding. A real transformer wraps the attention in a **transformer block** that adds three things:

### Feed-Forward Network (FFN)

The FFN is two matrix multiplies with a nonlinearity in between, applied to each token independently:

```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2

W1: [128, 512]    expand to 4× wider — a larger space to "think" in
W2: [512, 128]    compress back to original size
```

ReLU is just `max(0, x)` — it zeros out negatives. Without this nonlinearity two matrix multiplies in a row would collapse into one (since `(x @ A) @ B = x @ (AB)`), making the extra layer pointless.

Research has shown FFN layers are where transformers store factual knowledge — things like "Paris is the capital of France" live in these weights. Attention decides *what information to gather* from other tokens. The FFN then *processes what was gathered*. The more FFN capacity (wider layers, more blocks), the more knowledge the model can hold.

### Residual Connections

A residual connection adds the input back to the output:

```
output = layer(x) + x
```

This means each layer only needs to learn the *change* rather than the full representation from scratch. It also gives gradients a direct highway through the network during backpropagation, preventing the vanishing gradient problem.

### Layer Normalization

Layer norm normalizes each token's vector to have mean 0 and standard deviation 1, then applies learned scale (γ) and shift (β):

```
x_norm = (x - mean) / std
output = γ * x_norm + β
```

Without this, values drift to extreme ranges as they pass through layers, making training unstable.

### The Complete Block

```
x ──→ Attention ──→ + x (residual) ──→ Layer Norm ──→ FFN ──→ + (residual) ──→ Layer Norm ──→ out
```

Real GPT models stack many of these blocks (GPT-2 small: 12, GPT-3: 96). Each block refines the representation further — early blocks tend to learn syntax, middle blocks learn semantics, later blocks learn complex reasoning patterns. We use one block here since we are learning the mechanics. Stacking more is just repeating the same structure.

The backward pass for these new layers follows the same pattern as everything else — matrix multiply backward, chain rule through the normalization — we handle it all in the updated training code.

In [7]:
D_FFN = D_MODEL * 4  # 512 — the "wider space to think in"

# --- FFN weights ---
W1 = np.random.normal(0, 0.02, (D_MODEL, D_FFN))   # [128, 512] expand
b1 = np.zeros(D_FFN)                                 # [512]
W2 = np.random.normal(0, 0.02, (D_FFN, D_MODEL))   # [512, 128] compress back
b2 = np.zeros(D_MODEL)                               # [128]

# --- Layer norm parameters (learned scale and shift) ---
ln1_gamma = np.ones(D_MODEL)     # scale, starts at 1 (no change)
ln1_beta  = np.zeros(D_MODEL)    # shift, starts at 0 (no change)
ln2_gamma = np.ones(D_MODEL)
ln2_beta  = np.zeros(D_MODEL)

def layer_norm(x, gamma, beta, eps=1e-5):
    """Normalize each row to mean=0, std=1, then scale and shift."""
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    x_norm = (x - mean) / (std + eps)
    return gamma * x_norm + beta

def relu(x):
    """Zero out negatives."""
    return np.maximum(0, x)

def ffn(x, W1, b1, W2, b2):
    """Two matrix multiplies with ReLU in between."""
    hidden = relu(x @ W1 + b1)   # [seq_len, 512]
    return hidden @ W2 + b2      # [seq_len, 128]

# --- Demo: run the full transformer block on our attention output ---

# Step 1: residual connection around attention + layer norm
block_x = layer_norm(attention_output + final_embeddings, ln1_gamma, ln1_beta)

# Step 2: FFN + residual + layer norm
ffn_out = ffn(block_x, W1, b1, W2, b2)
block_out = layer_norm(ffn_out + block_x, ln2_gamma, ln2_beta)

print(f"Transformer Block walkthrough:")
print(f"  Input (final_embeddings):    {final_embeddings.shape}")
print(f"  After attention + residual:  {(attention_output + final_embeddings).shape}")
print(f"  After layer norm 1:          {block_x.shape}")
print(f"  FFN hidden layer:            [14, {D_FFN}]  (expanded 4×)")
print(f"  After FFN + residual:        {(ffn_out + block_x).shape}")
print(f"  After layer norm 2 (output): {block_out.shape}")

print(f"\nFFN weight shapes:")
print(f"  W1: {W1.shape}  (expand to {D_FFN})")
print(f"  W2: {W2.shape}  (compress back to {D_MODEL})")
print(f"  b1: {b1.shape}")
print(f"  b2: {b2.shape}")

print(f"\nLayer norm parameters: γ and β each [{D_MODEL}] (×2 norms)")

ffn_params = D_MODEL * D_FFN + D_FFN + D_FFN * D_MODEL + D_MODEL
ln_params = D_MODEL * 4  # 2 norms × (gamma + beta)
print(f"\nNew parameters:  FFN: {ffn_params:,}  LayerNorm: {ln_params:,}  Total new: {ffn_params + ln_params:,}")
print(f"Attention params: {total_params:,}")
print(f"Grand total:      {total_params + ffn_params + ln_params:,}")

Transformer Block walkthrough:
  Input (final_embeddings):    (14, 128)
  After attention + residual:  (14, 128)
  After layer norm 1:          (14, 128)
  FFN hidden layer:            [14, 512]  (expanded 4×)
  After FFN + residual:        (14, 128)
  After layer norm 2 (output): (14, 128)

FFN weight shapes:
  W1: (128, 512)  (expand to 512)
  W2: (512, 128)  (compress back to 128)
  b1: (512,)
  b2: (128,)

Layer norm parameters: γ and β each [128] (×2 norms)

New parameters:  FFN: 131,712  LayerNorm: 512  Total new: 132,224
Attention params: 65,536
Grand total:      197,760


## Training - Backpropagation

Training is just two steps repeated over and over:
1. **Forward pass** - run the input through the model and get a prediction (we already built this)
2. **Backward pass** - figure out how wrong we were and nudge every weight to be less wrong next time

### The Loss Function

First we need a number that says "how wrong was the prediction". We use **cross-entropy loss** which is simply:

```
loss = -log(probability the model gave to the correct token)
```

If the model put 90% probability on the right answer: loss = -log(0.9) = 0.105 (small, good)
If the model put 1% probability on the right answer: loss = -log(0.01) = 4.6 (big, bad)
If the model put 0.002% on the right answer (our random model): loss = -log(0.00002) = 10.8 (terrible)

The goal of training is to make this number go down.

### The Chain Rule - How Gradients Flow Backward

Every operation we did in the forward pass has a derivative. The chain rule says we can multiply these derivatives together to find out how much any weight contributed to the final error.

Each weight gets a **gradient** - a direction to move to reduce the loss. Then we update:

```
weight = weight - learning_rate * gradient
```

The learning rate is small (eg. 0.001) so we take tiny steps. Too big and we overshoot, too small and we take forever.

### Building Blocks

Before the walkthrough, these are the only derivative rules we need:

**Matrix multiply: `Y = X @ W`**
```
dW = Xᵀ @ dY       (gradient for the weight matrix — used to update it)
dX = dY @ Wᵀ       (gradient for the input — passed backward to the previous layer)
```

**Softmax + cross-entropy combined:**
```
dlogits = probs - one_hot(correct_token)
```
one_hot(correct token) is just an empty array with a 1 at the position of the correct token with our token Vocab dictionary

**ReLU: `Y = max(0, X)`**
```
dX = dY * (X > 0)      (gradient passes through where input was positive, zero where it was negative)
```

**Residual: `Y = layer(X) + X`**
```
dX = dlayer(X) + dY     (gradient flows through both the layer AND the skip connection)
```
This is why residuals help training — the gradient always has a direct path through the `+ X` even if the layer's gradient vanishes.

**Layer Norm:** The derivative involves the normalized values and the learned γ. It ensures gradients are properly scaled even when the forward values were large or small.

---

### Walking Backward Through the Forward Pass

Here is the full forward pass with the transformer block:
```
x = embedding + positional                          (1)
Q = x @ W_Q,  K = x @ W_K,  V = x @ W_V           (2)
scores = Q @ Kᵀ / √d                               (3)
weights = softmax(scores)                           (4)
attn_out = (weights @ V) → concat → @ W_O          (5)
x2 = layer_norm(attn_out + x)                      (6)  ← residual + norm
ffn_hidden = ReLU(x2 @ W1 + b1)                    (7)  ← FFN expand
ffn_out = ffn_hidden @ W2 + b2                      (8)  ← FFN compress
x3 = layer_norm(ffn_out + x2)                      (9)  ← residual + norm
logits = x3 @ W_vocab                              (10)
probs = softmax(logits)                             (11)
loss = -log(probs[correct])                         (12)
```

The backward pass walks through in reverse, same pattern at every step: compute `dW` (update the weight) and `dX` (pass backward).

---

### Causal Mask

Each token should only attend to tokens before it (and itself) so the model learns to predict from past context only. We set future positions to -infinity before softmax.

```
mask (4×4 example):
[[1, 0, 0, 0],       0 means "set score to -inf"
 [1, 1, 0, 0],       so after softmax these become 0.0
 [1, 1, 1, 0],
 [1, 1, 1, 1]]
```

This is purely a speed trick. Without the mask you would need to do a separate forward pass for each position (feed in 1 token, feed in 2 tokens, feed in 3 tokens...). The mask lets you do all positions in one pass because each token can only see tokens before it — identical to feeding in a shorter sequence. One sentence gives us 13 training examples for free instead of doing 13 separate passes.

In [8]:
LEARNING_RATE = 0.01

# Re-initialise all weights fresh for training
np.random.seed(42)
embedding_matrix = np.random.normal(0, 0.02, (VOCAB_SIZE, D_MODEL))
W_Q = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_K = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_V = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_O = np.random.normal(0, 0.02, (D_MODEL, D_MODEL))
W_vocab = np.random.normal(0, 0.02, (D_MODEL, VOCAB_SIZE))

# FFN weights
D_FFN = D_MODEL * 4
W1 = np.random.normal(0, 0.02, (D_MODEL, D_FFN))
b1 = np.zeros(D_FFN)
W2 = np.random.normal(0, 0.02, (D_FFN, D_MODEL))
b2 = np.zeros(D_MODEL)

# Layer norm parameters
ln1_gamma = np.ones(D_MODEL);  ln1_beta = np.zeros(D_MODEL)
ln2_gamma = np.ones(D_MODEL);  ln2_beta = np.zeros(D_MODEL)

# Training data: input tokens predict the next token at each position
input_tokens = tokens[:-1]
target_tokens = tokens[1:]

print(f"Training on: '{text}'")
print(f"  Input tokens:  {input_tokens}")
print(f"  Target tokens: {target_tokens}")
print(f"\nEach position predicts the next token:")
for i in range(len(input_tokens)):
    in_tok = tokenizer.decode(int(input_tokens[i]))
    tgt_tok = tokenizer.decode(int(target_tokens[i]))
    print(f"  '{in_tok}' → should predict '{tgt_tok}'")

Training on: 'Hello, world! This is ZachGPT learning to tokenize.'
  Input tokens:  [15496, 11, 995, 0, 770, 318, 18825, 38, 11571, 4673, 284, 11241, 1096]
  Target tokens: [11, 995, 0, 770, 318, 18825, 38, 11571, 4673, 284, 11241, 1096, 13]

Each position predicts the next token:
  'Hello' → should predict ','
  ',' → should predict ' world'
  ' world' → should predict '!'
  '!' → should predict ' This'
  ' This' → should predict ' is'
  ' is' → should predict ' Zach'
  ' Zach' → should predict 'G'
  'G' → should predict 'PT'
  'PT' → should predict ' learning'
  ' learning' → should predict ' to'
  ' to' → should predict ' token'
  ' token' → should predict 'ize'
  'ize' → should predict '.'


In [9]:
def layer_norm_forward(x, gamma, beta, eps=1e-5):
    """Forward pass for layer norm, returning intermediates for backward pass."""
    mean = x.mean(axis=-1, keepdims=True)
    var  = x.var(axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    out = gamma * x_norm + beta
    return out, x_norm, mean, var

def layer_norm_backward(dout, x_norm, var, gamma, eps=1e-5):
    """Backward pass for layer norm."""
    D = x_norm.shape[-1]
    dgamma = (dout * x_norm).sum(axis=0)
    dbeta  = dout.sum(axis=0)

    dx_norm = dout * gamma
    dvar  = (dx_norm * x_norm * -0.5 / (var + eps)).sum(axis=-1, keepdims=True)
    dmean = (-dx_norm / np.sqrt(var + eps)).sum(axis=-1, keepdims=True)
    dx = dx_norm / np.sqrt(var + eps) + 2 * dvar * x_norm * np.sqrt(var + eps) / D + dmean / D
    return dx, dgamma, dbeta

def train_step(input_toks, target_toks):
    """One full forward + backward pass with the complete transformer block."""
    global W1, b1, W2, b2, ln1_gamma, ln1_beta, ln2_gamma, ln2_beta
    seq_len = len(input_toks)

    # ==================== FORWARD PASS ====================

    # --- (1) Embedding + positional encoding ---
    tok_emb = embedding_matrix[input_toks]
    pos_enc = positional_matrix[:seq_len]
    x = tok_emb + pos_enc                               # [seq_len, 128]

    # --- Causal mask ---
    mask = np.tril(np.ones((seq_len, seq_len)))

    # --- (2-5) Multi-head attention ---
    heads_Q, heads_K, heads_V = [], [], []
    heads_scores, heads_weights, heads_out = [], [], []

    for h in range(N_HEADS):
        Q = x @ W_Q[h]
        K = x @ W_K[h]
        V = x @ W_V[h]
        scores = (Q @ K.T) / np.sqrt(D_HEAD)
        scores = np.where(mask == 1, scores, -1e9)
        weights = softmax(scores)
        head_out = weights @ V

        heads_Q.append(Q); heads_K.append(K); heads_V.append(V)
        heads_scores.append(scores); heads_weights.append(weights)
        heads_out.append(head_out)

    concatenated = np.concatenate(heads_out, axis=1)
    attn_out = concatenated @ W_O                        # [seq_len, 128]

    # --- (6) Residual + layer norm 1 ---
    res1 = attn_out + x
    x2, x2_norm, x2_mean, x2_var = layer_norm_forward(res1, ln1_gamma, ln1_beta)

    # --- (7-8) FFN ---
    ffn_pre = x2 @ W1 + b1                              # [seq_len, 512]
    ffn_hidden = np.maximum(0, ffn_pre)                  # ReLU
    ffn_out = ffn_hidden @ W2 + b2                       # [seq_len, 128]

    # --- (9) Residual + layer norm 2 ---
    res2 = ffn_out + x2
    x3, x3_norm, x3_mean, x3_var = layer_norm_forward(res2, ln2_gamma, ln2_beta)

    # --- (10-11) Project to vocab ---
    logits = x3 @ W_vocab                               # [seq_len, 50257]
    probs = softmax(logits)

    # --- (12) Loss ---
    loss = 0.0
    for t in range(seq_len):
        loss += -np.log(probs[t, target_toks[t]] + 1e-10)
    loss /= seq_len

    # ==================== BACKWARD PASS ====================

    # --- undo (11+12): softmax + cross-entropy ---
    dlogits = probs.copy()
    for t in range(seq_len):
        dlogits[t, target_toks[t]] -= 1
    dlogits /= seq_len

    # --- undo (10): logits = x3 @ W_vocab ---
    dW_vocab = x3.T @ dlogits
    dx3 = dlogits @ W_vocab.T

    # --- undo (9): layer norm 2 + residual ---
    dres2, dln2_gamma, dln2_beta = layer_norm_backward(dx3, x3_norm, x3_var, ln2_gamma)
    dffn_out = dres2
    dx2_from_res2 = dres2                                # residual: gradient flows through both paths

    # --- undo (8): ffn_out = ffn_hidden @ W2 + b2 ---
    dW2 = ffn_hidden.T @ dffn_out
    db2 = dffn_out.sum(axis=0)
    dffn_hidden = dffn_out @ W2.T

    # --- undo (7): ReLU + (x2 @ W1 + b1) ---
    dffn_pre = dffn_hidden * (ffn_pre > 0)               # ReLU backward
    dW1 = x2.T @ dffn_pre
    db1 = dffn_pre.sum(axis=0)
    dx2_from_ffn = dffn_pre @ W1.T

    # --- combine gradients flowing into x2 ---
    dx2 = dx2_from_ffn + dx2_from_res2

    # --- undo (6): layer norm 1 + residual ---
    dres1, dln1_gamma, dln1_beta = layer_norm_backward(dx2, x2_norm, x2_var, ln1_gamma)
    dattn_out = dres1
    dx_from_res1 = dres1                                 # residual skip connection

    # --- undo (5): attn_out = concatenated @ W_O ---
    dW_O = concatenated.T @ dattn_out
    dconcatenated = dattn_out @ W_O.T

    # --- undo (2-4) per head ---
    dx = np.zeros_like(x)

    for h in range(N_HEADS):
        dhead_out = dconcatenated[:, h*D_HEAD:(h+1)*D_HEAD]

        dweights = dhead_out @ heads_V[h].T
        dV = heads_weights[h].T @ dhead_out

        sum_dw = (dweights * heads_weights[h]).sum(axis=1, keepdims=True)
        dscores = heads_weights[h] * (dweights - sum_dw)
        dscores /= np.sqrt(D_HEAD)
        dscores = np.where(mask == 1, dscores, 0)

        dQ = dscores @ heads_K[h]
        dK = dscores.T @ heads_Q[h]

        W_Q[h] -= LEARNING_RATE * (x.T @ dQ)
        W_K[h] -= LEARNING_RATE * (x.T @ dK)
        W_V[h] -= LEARNING_RATE * (x.T @ dV)

        dx += dQ @ W_Q[h].T + dK @ W_K[h].T + dV @ W_V[h].T

    # --- add residual gradient from layer norm 1 ---
    dx += dx_from_res1

    # --- Update all weights ---
    W_O[:]  -= LEARNING_RATE * dW_O
    W_vocab[:] -= LEARNING_RATE * dW_vocab

    W1[:] -= LEARNING_RATE * dW1
    b1[:] -= LEARNING_RATE * db1
    W2[:] -= LEARNING_RATE * dW2
    b2[:] -= LEARNING_RATE * db2

    ln1_gamma -= LEARNING_RATE * dln1_gamma
    ln1_beta  -= LEARNING_RATE * dln1_beta
    ln2_gamma -= LEARNING_RATE * dln2_gamma
    ln2_beta  -= LEARNING_RATE * dln2_beta

    for t in range(seq_len):
        embedding_matrix[input_toks[t]] -= LEARNING_RATE * dx[t]

    return loss

# Test one step
loss = train_step(input_tokens, target_tokens)
print(f"Initial loss: {loss:.4f}")
print(f"Expected loss for random guessing over {VOCAB_SIZE} tokens: {np.log(VOCAB_SIZE):.4f}")
print(f"(These should be similar — the model is just guessing randomly)")

Initial loss: 10.9127
Expected loss for random guessing over 50257 tokens: 10.8249
(These should be similar — the model is just guessing randomly)


In [10]:
# --- Train for 1000 steps on our single sentence and watch the loss drop ---
losses = []

print(f"Training sentence: '{text}'")
print(f"Running 1000 steps of gradient descent...\n")

for step in range(1000):
    loss = train_step(input_tokens, target_tokens)
    losses.append(loss)
    if step % 100 == 0 or step == 999:
        print(f"Step {step:4d}  loss: {loss:.4f}")

print(f"\nLoss went from {losses[0]:.4f} → {losses[-1]:.4f}")
print(f"The model is memorising this one sentence (overfitting on purpose to prove training works)")

Training sentence: 'Hello, world! This is ZachGPT learning to tokenize.'
Running 1000 steps of gradient descent...

Step    0  loss: 10.8121
Step  100  loss: 1.5547
Step  200  loss: 0.6138
Step  300  loss: 0.3716
Step  400  loss: 0.2282
Step  500  loss: 0.1525
Step  600  loss: 0.1090
Step  700  loss: 0.0820
Step  800  loss: 0.0641
Step  900  loss: 0.0518
Step  999  loss: 0.0429

Loss went from 10.8121 → 0.0429
The model is memorising this one sentence (overfitting on purpose to prove training works)


In [11]:
# --- Now test predictions after training ---

def predict_next_trained(input_text):
    """Forward pass only, using the trained weights (full transformer block)."""
    toks = tokenizer.encode(input_text)
    seq_len = len(toks)

    tok_emb = embedding_matrix[toks]
    pos_enc = positional_matrix[:seq_len]
    x = tok_emb + pos_enc

    mask = np.tril(np.ones((seq_len, seq_len)))

    # Multi-head attention
    head_outputs = []
    for h in range(N_HEADS):
        Q = x @ W_Q[h]
        K = x @ W_K[h]
        V = x @ W_V[h]
        scores = (Q @ K.T) / np.sqrt(D_HEAD)
        scores = np.where(mask == 1, scores, -1e9)
        weights = softmax(scores)
        head_outputs.append(weights @ V)

    attn_out = np.concatenate(head_outputs, axis=1) @ W_O

    # Residual + layer norm 1
    x2 = layer_norm(attn_out + x, ln1_gamma, ln1_beta)

    # FFN + residual + layer norm 2
    ffn_hidden = np.maximum(0, x2 @ W1 + b1)
    ffn_out = ffn_hidden @ W2 + b2
    x3 = layer_norm(ffn_out + x2, ln2_gamma, ln2_beta)

    # Project to vocab
    logits = x3[-1] @ W_vocab
    probs = softmax(logits)

    top_5 = np.argsort(probs)[-5:][::-1]
    return [(tokenizer.decode(int(tid)), probs[tid]) for tid in top_5]

# Test on substrings of our training sentence - the model should know these
test_sentences = [
    "Hello, world! This is",
    "Hello, world! This is ZachGPT learning to",
    "The cat sat on the",
]

for sentence in test_sentences:
    predictions = predict_next_trained(sentence)
    print(f"Input: '{sentence}'")
    print(f"  Top 5 predictions:")
    for i, (token, prob) in enumerate(predictions):
        print(f"    {i+1}. '{token}' ({prob:.2%})")
    print()

Input: 'Hello, world! This is'
  Top 5 predictions:
    1. ' Zach' (95.24%)
    2. 'G' (2.47%)
    3. ' is' (2.07%)
    4. 'PT' (0.00%)
    5. ' This' (0.00%)

Input: 'Hello, world! This is ZachGPT learning to'
  Top 5 predictions:
    1. ' token' (95.13%)
    2. ' to' (2.36%)
    3. 'ize' (2.24%)
    4. ' learning' (0.00%)
    5. '.' (0.00%)

Input: 'The cat sat on the'
  Top 5 predictions:
    1. ' is' (93.89%)
    2. ' Zach' (3.02%)
    3. ' This' (2.87%)
    4. '!' (0.00%)
    5. 'G' (0.00%)



## Training on Real Data — Tiny Shakespeare

Everything above was trained on a single sentence to prove the mechanics work. Now we train on actual text — the complete works of Shakespeare (~1.1MB, ~300K tokens). This is where the model stops memorising and starts learning patterns in language.

### Two Problems to Solve First

**Problem 1: Vocabulary size**

Our GPT-2 tokenizer has 50,257 possible tokens, which means W_vocab is `[128, 50257]` and the embedding matrix is `[50257, 128]`. Every training step multiplies through these massive matrices even though Shakespeare only uses a few thousand of those tokens. The fix is simple: tokenize all of Shakespeare first, find which token IDs actually appear, and remap them to a compact 0..N range. Shakespeare uses roughly 4,000-5,000 unique tokens out of 50,257 so our matrices shrink by ~10×.

We keep the same GPT-2 tokenizer — we just maintain a mapping table so we can convert back when generating text.

**Problem 2: Sequence length**

We used `MAX_SEQ_LEN = 512` for demonstrations but training on 512-token sequences is slow. The attention scores matrix is `[seq_len, seq_len]` so halving the sequence length makes attention 4× faster. We use 128 tokens per chunk — enough context to learn sentence structure and common phrases while keeping each training step fast.

### Data Preparation

We tokenize all of Shakespeare into one long array of token IDs, remap to the compact vocabulary, then slice it into non-overlapping chunks of 128 tokens. Each chunk becomes one training example where position N predicts position N+1 (same as before, just more data).

In [12]:
# --- Load and prepare Shakespeare data ---
import os
import urllib.request

shakespeare_path = "../data/raw/tiny_shakespeare.txt"

if not os.path.exists(shakespeare_path):
    os.makedirs(os.path.dirname(shakespeare_path), exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print(f"Downloading Tiny Shakespeare from {url}...")
    urllib.request.urlretrieve(url, shakespeare_path)
    print("Done!")
else:
    print("Tiny Shakespeare already downloaded.")

with open(shakespeare_path, "r") as f:
    shakespeare_text = f.read()

print(f"Dataset size: {len(shakespeare_text):,} characters")
print(f"First 200 characters:\n{shakespeare_text[:200]}")

# Tokenize the entire text
all_tokens_gpt2 = tokenizer.encode(shakespeare_text)
print(f"\nTotal tokens: {len(all_tokens_gpt2):,}")

# --- Build compact vocabulary ---
# Find which GPT-2 token IDs actually appear in Shakespeare
unique_gpt2_ids = sorted(set(all_tokens_gpt2))
SHAKESPEARE_VOCAB_SIZE = len(unique_gpt2_ids)

# Create mappings: GPT-2 ID <-> compact ID
gpt2_to_compact = {gpt2_id: compact_id for compact_id, gpt2_id in enumerate(unique_gpt2_ids)}
compact_to_gpt2 = {compact_id: gpt2_id for gpt2_id, compact_id in gpt2_to_compact.items()}

# Remap all tokens to compact IDs
all_tokens = np.array([gpt2_to_compact[t] for t in all_tokens_gpt2])

print(f"\nVocabulary remapping:")
print(f"  GPT-2 vocab size:       {VOCAB_SIZE:,}")
print(f"  Shakespeare vocab size: {SHAKESPEARE_VOCAB_SIZE:,}")
print(f"  Reduction:              {VOCAB_SIZE / SHAKESPEARE_VOCAB_SIZE:.1f}×")

# Show some example remappings
print(f"\nExample remappings:")
for compact_id in [0, 1, 2, 100, 500, SHAKESPEARE_VOCAB_SIZE - 1]:
    gpt2_id = compact_to_gpt2[compact_id]
    token_text = tokenizer.decode(gpt2_id)
    print(f"  compact {compact_id:4d} ← GPT-2 {gpt2_id:5d} → '{token_text}'")

# --- Slice into training chunks ---
SEQ_LEN = 128

num_chunks = len(all_tokens) // (SEQ_LEN + 1)
chunks = []
for i in range(num_chunks):
    start = i * (SEQ_LEN + 1)
    chunk = all_tokens[start : start + SEQ_LEN + 1]
    chunks.append(chunk)

chunks = np.array(chunks)
print(f"\nTraining chunks:")
print(f"  Sequence length: {SEQ_LEN}")
print(f"  Number of chunks: {num_chunks:,}")
print(f"  Tokens used: {num_chunks * (SEQ_LEN + 1):,} / {len(all_tokens):,}")

# Show what a chunk looks like decoded
sample_chunk = chunks[0]
sample_text = tokenizer.decode([compact_to_gpt2[int(t)] for t in sample_chunk[:20]])
print(f"\nFirst chunk starts with: '{sample_text}...'")

Tiny Shakespeare already downloaded.
Dataset size: 1,115,394 characters
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you

Total tokens: 338,025

Vocabulary remapping:
  GPT-2 vocab size:       50,257
  Shakespeare vocab size: 11,706
  Reduction:              4.3×

Example remappings:
  compact    0 ← GPT-2     0 → '!'
  compact    1 ← GPT-2     3 → '$'
  compact    2 ← GPT-2     6 → '''
  compact  100 ← GPT-2   294 → ' th'
  compact  500 ← GPT-2   732 → 'we'
  compact 11705 ← GPT-2 50255 → ' gazed'

Training chunks:
  Sequence length: 128
  Number of chunks: 2,620
  Tokens used: 337,980 / 338,025

First chunk starts with: 'First Citizen:
Before we proceed any further, hear me speak.

All:
Spe...'


In [34]:
# --- Reinitialise all weights for the compact Shakespeare vocabulary ---
np.random.seed(42)

N_EPOCHS = 24
LR_SCHEDULE = [(5, 0.1), (5, 0.05), (4, 0.03), (4, 0.01), (3, 0.005), (3, 0.002)]
LEARNING_RATE = LR_SCHEDULE[0][1]

# Embedding and vocab projection now use the compact vocab size
embedding_matrix = np.random.normal(0, 0.02, (SHAKESPEARE_VOCAB_SIZE, D_MODEL))
W_vocab = np.random.normal(0, 0.02, (D_MODEL, SHAKESPEARE_VOCAB_SIZE))

# Attention weights (same as before)
W_Q = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_K = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_V = [np.random.normal(0, 0.02, (D_MODEL, D_HEAD)) for _ in range(N_HEADS)]
W_O = np.random.normal(0, 0.02, (D_MODEL, D_MODEL))

# FFN weights
D_FFN = D_MODEL * 4
W1 = np.random.normal(0, 0.02, (D_MODEL, D_FFN))
b1 = np.zeros(D_FFN)
W2 = np.random.normal(0, 0.02, (D_FFN, D_MODEL))
b2 = np.zeros(D_MODEL)

# Layer norm parameters
ln1_gamma = np.ones(D_MODEL);  ln1_beta = np.zeros(D_MODEL)
ln2_gamma = np.ones(D_MODEL);  ln2_beta = np.zeros(D_MODEL)

# Positional encoding for the new sequence length
positional_matrix = positional_encoding(SEQ_LEN, D_MODEL)

# Precompute causal mask
mask = np.tril(np.ones((SEQ_LEN, SEQ_LEN)))

# Count parameters
attn_params = N_HEADS * 3 * D_MODEL * D_HEAD + D_MODEL * D_MODEL
ffn_params = D_MODEL * D_FFN + D_FFN + D_FFN * D_MODEL + D_MODEL
ln_params = D_MODEL * 4
embed_params = SHAKESPEARE_VOCAB_SIZE * D_MODEL
vocab_params = D_MODEL * SHAKESPEARE_VOCAB_SIZE
total = attn_params + ffn_params + ln_params + embed_params + vocab_params

print(f"Shakespeare training setup:")
print(f"  Vocab size:      {SHAKESPEARE_VOCAB_SIZE:,} (was {VOCAB_SIZE:,})")
print(f"  Sequence length: {SEQ_LEN} (was 512)")
print(f"  Epochs:          {N_EPOCHS}")
print(f"  LR schedule:     {LR_SCHEDULE}")
print(f"  Chunks per epoch: {len(chunks):,}")
print(f"\nParameter counts:")
print(f"  Embeddings:  {embed_params:,}")
print(f"  Attention:   {attn_params:,}")
print(f"  FFN:         {ffn_params:,}")
print(f"  Layer norm:  {ln_params:,}")
print(f"  W_vocab:     {vocab_params:,}")
print(f"  Total:       {total:,}")
print(f"\nW_vocab size: [{D_MODEL}, {SHAKESPEARE_VOCAB_SIZE}] vs old [{D_MODEL}, {VOCAB_SIZE}]")
print(f"  That is {VOCAB_SIZE * D_MODEL / (SHAKESPEARE_VOCAB_SIZE * D_MODEL):.1f}× smaller — this is where most of the speedup comes from")

Shakespeare training setup:
  Vocab size:      11,706 (was 50,257)
  Sequence length: 128 (was 512)
  Epochs:          24
  LR schedule:     [(5, 0.1), (5, 0.05), (4, 0.03), (4, 0.01), (3, 0.005), (3, 0.002)]
  Chunks per epoch: 2,620

Parameter counts:
  Embeddings:  1,498,368
  Attention:   65,536
  FFN:         131,712
  Layer norm:  512
  W_vocab:     1,498,368
  Total:       3,194,496

W_vocab size: [128, 11706] vs old [128, 50257]
  That is 4.3× smaller — this is where most of the speedup comes from


In [35]:
# --- Train on Shakespeare with learning rate schedule ---

all_losses = []

# Build a flat list: epoch -> learning rate
lr_by_epoch = []
for n_epochs, lr in LR_SCHEDULE:
    lr_by_epoch.extend([lr] * n_epochs)

for epoch in range(N_EPOCHS):
    LEARNING_RATE = lr_by_epoch[epoch]
    epoch_losses = []
    epoch_start = time.time()

    shuffle_idx = np.random.permutation(len(chunks))

    for i, chunk_idx in enumerate(shuffle_idx):
        chunk = chunks[chunk_idx]
        input_toks = chunk[:-1]
        target_toks = chunk[1:]

        loss = train_step(input_toks, target_toks)
        epoch_losses.append(loss)

        if (i + 1) % 500 == 0:
            avg_loss = np.mean(epoch_losses[-500:])
            elapsed = time.time() - epoch_start
            steps_per_sec = (i + 1) / elapsed
            print(f"  Epoch {epoch+1}/{N_EPOCHS}  step {i+1:,}/{len(chunks):,}  "
                  f"loss: {avg_loss:.4f}  lr: {LEARNING_RATE}  ({steps_per_sec:.1f} steps/sec)")

    avg = np.mean(epoch_losses)
    elapsed = time.time() - epoch_start
    all_losses.extend(epoch_losses)
    print(f"Epoch {epoch+1}/{N_EPOCHS} complete — avg loss: {avg:.4f}  lr: {LEARNING_RATE}  "
          f"time: {elapsed:.0f}s  ({len(chunks)/elapsed:.1f} steps/sec)")

print(f"\nTraining complete. Final average loss: {np.mean(all_losses[-500:]):.4f}")
print(f"Random guessing loss would be: {np.log(SHAKESPEARE_VOCAB_SIZE):.4f}")

  Epoch 1/24  step 500/2,620  loss: 6.8172  lr: 0.1  (87.2 steps/sec)
  Epoch 1/24  step 1,000/2,620  loss: 6.4696  lr: 0.1  (86.3 steps/sec)
  Epoch 1/24  step 1,500/2,620  loss: 6.3891  lr: 0.1  (86.4 steps/sec)
  Epoch 1/24  step 2,000/2,620  loss: 6.2309  lr: 0.1  (85.8 steps/sec)
  Epoch 1/24  step 2,500/2,620  loss: 6.0943  lr: 0.1  (85.9 steps/sec)
Epoch 1/24 complete — avg loss: 6.3795  lr: 0.1  time: 31s  (85.8 steps/sec)
  Epoch 2/24  step 500/2,620  loss: 5.9170  lr: 0.1  (86.3 steps/sec)
  Epoch 2/24  step 1,000/2,620  loss: 5.8534  lr: 0.1  (86.3 steps/sec)
  Epoch 2/24  step 1,500/2,620  loss: 5.8310  lr: 0.1  (86.3 steps/sec)
  Epoch 2/24  step 2,000/2,620  loss: 5.7076  lr: 0.1  (86.2 steps/sec)
  Epoch 2/24  step 2,500/2,620  loss: 5.8586  lr: 0.1  (86.2 steps/sec)
Epoch 2/24 complete — avg loss: 5.8279  lr: 0.1  time: 30s  (86.1 steps/sec)
  Epoch 3/24  step 500/2,620  loss: 5.9054  lr: 0.1  (86.0 steps/sec)
  Epoch 3/24  step 1,000/2,620  loss: 5.5534  lr: 0.1  (83.5

In [29]:
# --- Generate Shakespeare-style text ---

def generate(prompt, max_tokens=200, temperature=0.8):
    """Generate text token by token using the trained model."""
    # Tokenize prompt and remap to compact IDs
    gpt2_toks = tokenizer.encode(prompt)
    toks = []
    for t in gpt2_toks:
        if t in gpt2_to_compact:
            toks.append(gpt2_to_compact[t])
        else:
            toks.append(0)

    generated_compact = list(toks)

    for _ in range(max_tokens):
        # Only use the last SEQ_LEN tokens as context
        context = generated_compact[-SEQ_LEN:]
        seq_len = len(context)

        tok_emb = embedding_matrix[context]
        pos_enc = positional_matrix[:seq_len]
        x = tok_emb + pos_enc

        m = np.tril(np.ones((seq_len, seq_len)))

        head_outputs = []
        for h in range(N_HEADS):
            Q = x @ W_Q[h]
            K = x @ W_K[h]
            V = x @ W_V[h]
            scores = (Q @ K.T) / np.sqrt(D_HEAD)
            scores = np.where(m == 1, scores, -1e9)
            weights = softmax(scores)
            head_outputs.append(weights @ V)

        attn_out = np.concatenate(head_outputs, axis=1) @ W_O
        x2 = layer_norm(attn_out + x, ln1_gamma, ln1_beta)
        ffn_hidden = np.maximum(0, x2 @ W1 + b1)
        ffn_out = ffn_hidden @ W2 + b2
        x3 = layer_norm(ffn_out + x2, ln2_gamma, ln2_beta)

        logits = x3[-1] @ W_vocab

        # Temperature sampling: lower = more confident, higher = more random
        logits = logits / temperature
        probs = softmax(logits)

        # Sample from the distribution rather than always picking the top token
        next_token = np.random.choice(len(probs), p=probs)
        generated_compact.append(next_token)

    # Convert compact IDs back to GPT-2 IDs and decode
    gpt2_ids = [compact_to_gpt2[t] for t in generated_compact]
    return tokenizer.decode(gpt2_ids)

# Generate some text!
prompts = [
    "First Citizen:\n",
    "ROMEO:\nO, ",
    "To be or not to be",
]

for prompt in prompts:
    print(f"{'='*60}")
    print(f"Prompt: {repr(prompt)}")
    print(f"{'='*60}")
    print(generate(prompt, max_tokens=150, temperature=0.8))
    print()

Prompt: 'First Citizen:\n'
First Citizen:
To yours.
LUCENTIO:
And make twisa, so I will have,
I may best being a man,
That well, be the whole father,
And not leave you, my books and that;
But that presence that you may have possess'd,

Nay, to you not; therefore, proud,
ROMEO:
INGHAM:

LADYORK:
LADY ANNE:
ROMEO:
HORTENSIO:

And that hope; then, what I pray thee so elder.
ROMEO:



Which he will I will see for the door, and,
TRANUS:

And do he were

Prompt: 'ROMEO:\nO, '
ROMEO:
O,  could not the king.



I did beseech you:

From my lord, I do your husband;
And all a man?
Thou not, if you be so:

From him:
CATESBY:

LUCENTIO: but love.
Which is love!
That her.
ELO:
We have you, sir?
The town, and all your hands,

DUKE OF YORK:
LADYORK:

The breath.
Clown:
Marry, while you are you, and her.

HORTENSIO:
Lead-morrow well, put thyself!
Where, and ever never

Prompt: 'To be or not to be'
To be or not to be
Which brought it.
SICINIUS:

But as a king, that I seems.
The fool, my lord, nor done; 